In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch.backends.cudnn as cudnn

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
print(f"PyTorch Version: {torch.__version__}")

# --- USER CONFIGURATION ---
SELECTED_GPU_INDEX = 0  # Adjust based on your nvidia-smi
DATASET_ROOT = "./NIRData"
BATCH_SIZE = 64      
LEARNING_RATE = 2e-4 
NUM_EPOCHS = 100      
IMG_SIZE = 256       
# --------------------------

if torch.cuda.is_available():
    device = torch.device(f"cuda:{SELECTED_GPU_INDEX}")
    cudnn.benchmark = True
    print(f"✅ Using GPU: {torch.cuda.get_device_name(SELECTED_GPU_INDEX)}")
else:
    device = torch.device("cpu")
    print("⚠️ Warning: CUDA not found. Using CPU.")
    
torch.manual_seed(42)

# ==========================================
# 2. DATASET & DATALOADERS
# ==========================================
class CapsicumDataset(Dataset):
    def __init__(self, root_dir, subdir_a, subdir_b, transform=None, return_filename=False):
        self.dir_a = os.path.join(root_dir, subdir_a)
        self.dir_b = os.path.join(root_dir, subdir_b)
        self.transform = transform
        self.return_filename = return_filename
        
        self.image_filenames = sorted([
            f for f in os.listdir(self.dir_a) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        path_a = os.path.join(self.dir_a, img_name)
        path_b = os.path.join(self.dir_b, img_name)

        img_a = Image.open(path_a).convert("RGB")
        img_b = Image.open(path_b).convert("L") 
        
        if self.transform:
            img_a = self.transform(img_a)
            img_b = self.transform(img_b)

        if self.return_filename:
            return img_a, img_b, img_name
        return img_a, img_b

transform_pipeline = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = CapsicumDataset(DATASET_ROOT, 'train_A', 'train_B', transform=transform_pipeline)
val_dataset = CapsicumDataset(DATASET_ROOT, 'val_A', 'val_B', transform=transform_pipeline)
test_dataset = CapsicumDataset(DATASET_ROOT, 'test_A', 'test_B', transform=transform_pipeline, return_filename=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ==========================================
# 3. ARCHITECTURES
# ==========================================
def conv_block(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True)
    )

class StandardUNet(nn.Module):
    def __init__(self):
        super(StandardUNet, self).__init__()
        self.e1 = conv_block(3, 64)
        self.pool = nn.MaxPool2d(2)
        self.e2 = conv_block(64, 128)
        self.e3 = conv_block(128, 256)
        self.e4 = conv_block(256, 512)
        self.b = conv_block(512, 1024)

        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.d1 = conv_block(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.d2 = conv_block(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.d3 = conv_block(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.d4 = conv_block(128, 64)
        self.out = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        c1 = self.e1(x)
        c2 = self.e2(self.pool(c1))
        c3 = self.e3(self.pool(c2))
        c4 = self.e4(self.pool(c3))
        b = self.b(self.pool(c4))

        d1 = self.d1(torch.cat((self.up1(b), c4), dim=1))
        d2 = self.d2(torch.cat((self.up2(d1), c3), dim=1))
        d3 = self.d3(torch.cat((self.up3(d2), c2), dim=1))
        d4 = self.d4(torch.cat((self.up4(d3), c1), dim=1))
        return self.out(d4)

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, kernel_size=1, padding=0), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, kernel_size=1, padding=0), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, kernel_size=1, padding=0), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        return x * self.psi(psi)

class AttentionUNet(nn.Module):
    def __init__(self):
        super(AttentionUNet, self).__init__()
        self.e1 = conv_block(3, 64)
        self.pool = nn.MaxPool2d(2)
        self.e2 = conv_block(64, 128)
        self.e3 = conv_block(128, 256)
        self.e4 = conv_block(256, 512)
        self.b = conv_block(512, 1024)

        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.att1 = AttentionBlock(F_g=512, F_l=512, F_int=256)
        self.d1 = conv_block(1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.att2 = AttentionBlock(F_g=256, F_l=256, F_int=128)
        self.d2 = conv_block(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.att3 = AttentionBlock(F_g=128, F_l=128, F_int=64)
        self.d3 = conv_block(256, 128)

        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.att4 = AttentionBlock(F_g=64, F_l=64, F_int=32)
        self.d4 = conv_block(128, 64)
        self.out = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        c1 = self.e1(x)
        c2 = self.e2(self.pool(c1))
        c3 = self.e3(self.pool(c2))
        c4 = self.e4(self.pool(c3))
        b = self.b(self.pool(c4))

        u1 = self.up1(b)
        d1 = self.d1(torch.cat((u1, self.att1(g=u1, x=c4)), dim=1))
        
        u2 = self.up2(d1)
        d2 = self.d2(torch.cat((u2, self.att2(g=u2, x=c3)), dim=1))
        
        u3 = self.up3(d2)
        d3 = self.d3(torch.cat((u3, self.att3(g=u3, x=c2)), dim=1))
        
        u4 = self.up4(d3)
        d4 = self.d4(torch.cat((u4, self.att4(g=u4, x=c1)), dim=1))
        
        return self.out(d4)

# ==========================================
# 4. LOSS FUNCTIONS & UTILS
# ==========================================
class SSIMLoss(nn.Module):
    def __init__(self):
        super(SSIMLoss, self).__init__()
    def forward(self, img1, img2):
        mu1, mu2 = nn.functional.avg_pool2d(img1, 3, 1, 1), nn.functional.avg_pool2d(img2, 3, 1, 1)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        sigma1_sq = nn.functional.avg_pool2d(img1 * img1, 3, 1, 1) - mu1_sq
        sigma2_sq = nn.functional.avg_pool2d(img2 * img2, 3, 1, 1) - mu2_sq
        sigma12 = nn.functional.avg_pool2d(img1 * img2, 3, 1, 1) - mu1_mu2
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
        return 1 - ssim_map.mean()

class MixedLoss(nn.Module):
    def __init__(self, alpha=0.85):
        super(MixedLoss, self).__init__()
        self.alpha = alpha
        self.l1 = nn.L1Loss()
        self.ssim = SSIMLoss()
    def forward(self, pred, target):
        return self.alpha * self.l1(pred, target) + (1 - self.alpha) * self.ssim(pred, target)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ==========================================
# 5. VISUALIZATION
# ==========================================
def visualize_paper_results(rgb, real_nir, pred_nir, filename, metrics, model_name):
    rgb_np = rgb.permute(1, 2, 0).cpu().numpy()
    real_np = real_nir.squeeze().cpu().numpy()
    pred_np = pred_nir.squeeze().cpu().numpy()
    error_map = np.abs(real_np - pred_np)
    
    fig = plt.figure(figsize=(20, 4))
    fig.suptitle(f"Model: {model_name} | Image: {filename}", fontsize=14, fontweight='bold')
    
    plt.subplot(1, 5, 1)
    plt.imshow(rgb_np)
    plt.title("Input RGB", fontsize=10)
    plt.axis('off')
    
    plt.subplot(1, 5, 2)
    plt.imshow(real_np, cmap='gray', vmin=0, vmax=1)
    plt.title("Ground Truth NIR", fontsize=10)
    plt.axis('off')
    
    plt.subplot(1, 5, 3)
    plt.imshow(pred_np, cmap='gray', vmin=0, vmax=1)
    plt.title(f"Predicted NIR\nPSNR: {metrics['psnr']:.2f} | SSIM: {metrics['ssim']:.3f}", fontsize=10)
    plt.axis('off')
    
    plt.subplot(1, 5, 4)
    im = plt.imshow(error_map, cmap='jet', vmin=0, vmax=0.5) 
    plt.title(f"Absolute Error\nMAE: {metrics['mae']:.4f}", fontsize=10)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.axis('off')
    
    plt.subplot(1, 5, 5)
    plt.hist2d(real_np.flatten(), pred_np.flatten(), bins=100, cmap='viridis', norm=mcolors.LogNorm())
    plt.plot([0, 1], [0, 1], 'r--', lw=2)
    plt.title(f"Intensity Distribution\nRMSE: {metrics['rmse']:.4f}", fontsize=10)
    plt.xlabel('Ground Truth')
    plt.ylabel('Prediction')
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 6. PIPELINE FUNCTIONS
# ==========================================
def train_model(model, model_name, save_path):
    print(f"\n🚀 Starting Training: {model_name}")
    print(f"   Trainable Parameters: {count_parameters(model):,}")
    
    criterion = MixedLoss(alpha=0.85).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    
    best_val_loss = float('inf')
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_train_loss = 0.0
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)
        
        for rgb, nir in train_loop:
            rgb, nir = rgb.to(device), nir.to(device)
            optimizer.zero_grad()
            loss = criterion(model(rgb), nir)
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()
            
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for rgb, nir in val_loader:
                rgb, nir = rgb.to(device), nir.to(device)
                running_val_loss += criterion(model(rgb), nir).item()
                
        avg_train = running_train_loss / len(train_loader)
        avg_val = running_val_loss / len(val_loader)
        scheduler.step()
        
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), save_path)
            
    print(f"✅ Training Complete for {model_name}. Best Val Loss: {best_val_loss:.4f}")

def test_and_visualize(model, model_name, save_path, output_dir):
    print(f"\n⚡ Testing: {model_name}")
    os.makedirs(output_dir, exist_ok=True)
    
    model.load_state_dict(torch.load(save_path))
    model.eval()
    
    metrics = {'psnr': 0.0, 'ssim': 0.0, 'mae': 0.0, 'rmse': 0.0}
    count = 0
    
    test_loop = tqdm(test_loader, desc="Evaluating", unit="img")
    with torch.no_grad():
        for rgb, real_nir, fname_tuple in test_loop:
            rgb, real_nir = rgb.to(device), real_nir.to(device)
            filename = fname_tuple[0]
            
            pred_nir = torch.clamp(model(rgb), 0, 1)
            save_image(pred_nir, os.path.join(output_dir, filename))
            
            real_np, pred_np = real_nir.squeeze().cpu().numpy(), pred_nir.squeeze().cpu().numpy()
            
            cur_m = {
                'psnr': psnr_metric(real_np, pred_np, data_range=1.0),
                'ssim': ssim_metric(real_np, pred_np, data_range=1.0),
                'mae': mean_absolute_error(real_np.flatten(), pred_np.flatten()),
                'rmse': math.sqrt(mean_squared_error(real_np.flatten(), pred_np.flatten()))
            }
            
            for k in metrics: metrics[k] += cur_m[k]
            count += 1
            
            if count <= 2: # Visualize first 2 samples
                visualize_paper_results(rgb[0], real_nir[0], pred_nir[0], filename, cur_m, model_name)

    print("\n" + "="*40)
    print(f"📊 FINAL RESULTS: {model_name}")
    for k, v in metrics.items(): print(f"   {k.upper()}: {v/count:.4f}")
    print("="*40)

# ==========================================
# 7. RUN THE STUDY
# ==========================================

# 1. Standard U-Net Baseline
std_model = StandardUNet().to(device)
std_weights = "best_standard_unet.pth"
std_output = "./results_standard_unet"

train_model(std_model, "Standard U-Net", std_weights)
test_and_visualize(std_model, "Standard U-Net", std_weights, std_output)

# 2. Attention U-Net Model
att_model = AttentionUNet().to(device)
att_weights = "best_attention_unet.pth"
att_output = "./results_attention_unet"

train_model(att_model, "Attention U-Net", att_weights)
test_and_visualize(att_model, "Attention U-Net", att_weights, att_output)